# haruihi_cleaner

Example of dataset: `{"user_role": "旅行者", "user_question": "「这个模型是我亲手制作的,送给你。」", "agent_role": "雷电将军", "agent_response": "「哦？这是你亲手制作的模型吗？看起来相当精致。你的手艺不错嘛。谢谢你的礼物，我会好好珍藏的。」", "question_source": "story", "more_dialogues": [], "agent_role_name_en": "raidenShogun"}` 

In [2]:
# get current cwd.

import os
os.getcwd()

'c:\\Users\\Muika\\Desktop\\Paper\\src\\dataset'

In [ ]:
# get raw dataset path

from pathlib import Path

raw_dataset_path = Path(r"..\..\Dataset\Haruhi\Haruhi_54K_v1.jsonl")
raw_dataset_path.is_file()

True

In [35]:
# Load and Clean the Dataset
# {"user_role": "旅行者", "user_question": "「我特意准备的点心,尝一尝吧。」", "agent_role": "雷电将军", "agent_response": "「哦？你特意为我准备的点心吗？看来你对我很了解啊。好吧，我就尝一尝。」", "question_source": "story", "more_dialogues": [], "agent_role_name_en": "raidenShogun"}

from pathlib import Path
import json

MIN_RESPONSE_LEN = 10

raw_dataset_path = Path(r"..\..\Dataset\Haruhi\Haruhi_54K_v1.jsonl")
output_dataset_path = Path(r"..\..\Dataset\Haruhi\Haruhi_clean.jsonl")

lines = raw_dataset_path.read_text(encoding="utf-8").split("\n")
datasets: list[dict[str,str]] = []
user_roles: list[str] = []
agent_roles: list[str] = []

for line in lines:
    if not line:
        continue        
    data = json.loads(line)
    datasets.append(data)
    user_roles.append(data["user_role"])
    agent_roles.append(data["agent_role"])

user_roles_set = set(user_roles)
agent_roles_set = set(agent_roles)

print("user_roles: ", len(user_roles_set))
print("agent_roles: ", len(agent_roles_set))

anime_datasets = []

for item in datasets:
    if item["agent_role"] in ["钟离", "旅行者", "神里绫华", "派蒙", "凉宫", "胡桃", "春日"]:
        item["user_question"] = item["user_question"].removeprefix("「").removesuffix("」").replace("<br>", "")
        item["agent_response"] = item["agent_response"].removeprefix("「").removesuffix("」").replace("<br>", "")
        if len(item["agent_response"]) < MIN_RESPONSE_LEN:
            continue
        if item["more_dialogues"]:
            pass
            # print(item["more_dialogues"])
            
        anime_datasets.append(item)

output_dataset_path.write_text("\n".join([json.dumps(item, ensure_ascii=False) for item in anime_datasets]), encoding="utf-8")

# union_roles: 
# user_roles: 旅行者 派蒙 凉宫
# agent_roles: 

user_roles:  1675
agent_roles:  299


1181797

In [36]:
import json
from typing import List, Dict, cast
from pathlib import Path
import logging

SFT_OUTPUT = Path(r"..\..\Dataset\Haruhi\Haruhi_sft.jsonl")

def convert_entry_to_sft(entry: Dict) -> Dict[str, list[str]]:
    """Convert a single entry (with possible more_dialogues) to SFT-style messages"""
    messages = []

    # Add system prompt
    agent_name = entry.get("agent_role", "角色")
    system_msg = f"你是{agent_name}，请根据上下文自然对话。"
    messages.append({"role": "system", "content": system_msg})

    # 初始对话
    # user_name = entry.get("user_role", "用户")
    user_msg = entry['user_question']
    messages.append({"role": "user", "content": user_msg})

    messages.append({"role": "assistant", "content": entry['agent_response']})

    user_text_cache = ""

    # 追加 more_dialogues（交替推断说话者）
    for index, dialogue in enumerate(entry.get("more_dialogues", [])):
        dialogue = cast(str, dialogue)

        if ":" in dialogue:
            speaker, content = dialogue.split(":", 1)
            content = content.removeprefix("「").removesuffix("」").replace("<br>", "")
            if speaker.strip() == agent_name:
                messages.append({"role": "user", "content": user_text_cache})
                messages.append({"role": "assistant", "content": f"{content}"})
                user_text_cache = ""
            else:
                speaker = speaker or "旁白"
                user_text_cache += f"{speaker}: {content} "
        else:
            # fallback
            messages.append({"role": "user", "content": dialogue})

    return {"messages": messages}


def convert_dataset(input_path: Path, output_path: Path):
    """Convert entire JSONL dataset"""
    with open(input_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    output_data = []

    for line in lines:
        if not line.strip():
            continue
        entry = json.loads(line)
        sft_item = convert_entry_to_sft(entry)

        sft_item = cast(dict[str, dict[str, str]], sft_item)
        if len(sft_item["messages"]) % 2 == 0:
            print(sft_item)
            continue

        output_data.append(sft_item)

    # 写入结果
    with open(output_path, "w", encoding="utf-8") as f:
        for item in output_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"转换完成，共生成 {len(output_data)} 条样本，输出至 {output_path}")


if __name__ == "__main__":

    convert_dataset(output_dataset_path, SFT_OUTPUT)


转换完成，共生成 3906 条样本，输出至 ..\..\Dataset\Haruhi\Haruhi_sft.jsonl
